# Phase 3 - Task 2: FRED Macroeconomic Data Integration

Enhance the Lending Club default-prediction use cases by joining **Federal Reserve Economic Data (FRED)** with our Silver/Gold loan tables. The hypothesis is simple and economically well-grounded: **borrower defaults depend on macro conditions at the time the loan is issued**. Loans issued during rising-unemployment or tightening-rate regimes should default at higher rates than loans issued during expansions, even after controlling for borrower-level features.

## Data source
FRED (Federal Reserve Bank of St. Louis), public CSV endpoints, no auth required.

| Series ID | What it measures | Frequency | Join key |
|---|---|---|---|
| `UNRATE` | US civilian unemployment rate | monthly | `year_month` |
| `FEDFUNDS` | Effective federal funds rate | monthly | `year_month` |
| `CPIAUCSL` | Consumer Price Index (all urban, SA) | monthly | `year_month` |
| `CAUR`, `TXUR`, `NYUR`, `FLUR`, `ILUR` | State unemployment rates (top 5 LC-volume states) | monthly | `year_month` + `addr_state` |

Access URL template: `https://fred.stlouisfed.org/graph/fredgraph.csv?id=<ID>`

## Flow

1. Fetch FRED CSVs -> Spark DataFrame -> Delta `bronze_macro`
2. Clean + pivot to `silver_macro` (one row per year_month with all series as columns)
3. Join with `silver_loans` on `year_month` (and state where applicable) -> `gold_loans_macro`
4. Three data insights (visualized)
5. Retrain the logistic regression with macro features; compare to the loan-features-only baseline

## Use-case alignment (Phase 1)
- **Investor risk scoring** - macro features let investors stress-test defaults against rate/unemployment regimes
- **Originator underwriting adjustment** - tighten or loosen credit policy based on current macro conditions
- **Regulatory stress testing** - the model now has explicit macro sensitivity

In [ ]:
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("phase3-macro").getOrCreate()

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests, io

SILVER_LOANS_TABLE = "workspace.default.silver_loans"
BRONZE_MACRO_TABLE = "workspace.default.bronze_macro"
SILVER_MACRO_TABLE = "workspace.default.silver_macro"
GOLD_COMBINED_TABLE = "workspace.default.gold_loans_macro"

## Step 1 - Fetch FRED series

In [ ]:
NATIONAL_SERIES = ["UNRATE", "FEDFUNDS", "CPIAUCSL"]
STATE_SERIES = {"CA": "CAUR", "TX": "TXUR", "NY": "NYUR", "FL": "FLUR", "IL": "ILUR"}

def fetch_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text))
    df.columns = ["observation_date", "value"]
    df["series_id"] = series_id
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df[["series_id", "observation_date", "value"]]

all_ids = NATIONAL_SERIES + list(STATE_SERIES.values())
dfs = [fetch_fred(sid) for sid in all_ids]
macro_pd = pd.concat(dfs, ignore_index=True)
print(f"Fetched {len(all_ids)} series, {len(macro_pd):,} total rows")
macro_pd.groupby("series_id").agg(rows=("value", "size"),
                                  start=("observation_date", "min"),
                                  end=("observation_date", "max"))

## Step 2 - Bronze: persist raw FRED data as Delta

In [ ]:
macro_sdf = spark.createDataFrame(macro_pd)
(
    macro_sdf.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_MACRO_TABLE)
)
print(f"Wrote {BRONZE_MACRO_TABLE}")

## Step 3 - Silver: pivot to year_month + series columns

One row per `year_month` with columns for each macro series. State unemployment rates stay as separate columns; the join step looks them up by borrower state.

In [ ]:
bronze = spark.table(BRONZE_MACRO_TABLE)

bronze = bronze.withColumn(
    "year_month",
    F.date_trunc("month", F.col("observation_date")).cast("date"),
)

silver = bronze.groupBy("year_month").pivot("series_id").agg(F.first("value"))

# CPI YOY - 12-month change in CPI -> inflation rate proxy
from pyspark.sql.window import Window
w = Window.orderBy("year_month")
silver = silver.withColumn(
    "CPI_YOY",
    (F.col("CPIAUCSL") / F.lag("CPIAUCSL", 12).over(w) - 1) * 100,
)

(
    silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_MACRO_TABLE)
)
print(f"Wrote {SILVER_MACRO_TABLE}")
spark.table(SILVER_MACRO_TABLE).orderBy(F.desc("year_month")).limit(3).toPandas()

## Step 4 - Gold: join macro features with loans

Join on `year_month` (national series) and look up the state-specific unemployment rate via a CASE expression on `addr_state`. Falls back to the national rate for states outside our top 5.

In [ ]:
loans = spark.table(SILVER_LOANS_TABLE)
loans = loans.withColumn(
    "year_month",
    F.date_trunc("month", F.col("issue_d")).cast("date"),
)

macro = spark.table(SILVER_MACRO_TABLE)

combined = loans.join(macro, on="year_month", how="left")

state_lookup = (
    F.when(F.col("addr_state") == "CA", F.col("CAUR"))
     .when(F.col("addr_state") == "TX", F.col("TXUR"))
     .when(F.col("addr_state") == "NY", F.col("NYUR"))
     .when(F.col("addr_state") == "FL", F.col("FLUR"))
     .when(F.col("addr_state") == "IL", F.col("ILUR"))
     .otherwise(F.col("UNRATE"))
)
combined = combined.withColumn("state_unrate", state_lookup)

# real_int_rate: borrower rate minus inflation
combined = combined.withColumn("real_int_rate", F.col("int_rate") - F.col("CPI_YOY"))
# rate_spread: borrower rate vs risk-free fed funds rate
combined = combined.withColumn("rate_spread", F.col("int_rate") - F.col("FEDFUNDS"))

combined = combined.drop("CAUR", "TXUR", "NYUR", "FLUR", "ILUR")

(
    combined.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_COMBINED_TABLE)
)
print(f"Wrote {GOLD_COMBINED_TABLE}: {spark.table(GOLD_COMBINED_TABLE).count():,} rows")

## Insight 1 - National unemployment rate at loan issuance vs default rate

Bin all loans by the unemployment rate at their issue month and plot charge-off frequency.

In [ ]:
q1 = spark.sql(f"""
SELECT
    ROUND(UNRATE, 1) AS unrate_bin,
    COUNT(*)          AS n_loans,
    AVG(charged_off)  AS default_rate
FROM {GOLD_COMBINED_TABLE}
WHERE UNRATE IS NOT NULL
GROUP BY ROUND(UNRATE, 1)
ORDER BY unrate_bin
""").toPandas()

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar(q1["unrate_bin"], q1["n_loans"], width=0.08, alpha=0.3, color="steelblue", label="# loans")
ax1.set_ylabel("# loans", color="steelblue")
ax2 = ax1.twinx()
ax2.plot(q1["unrate_bin"], q1["default_rate"], "o-", color="darkred", lw=2, label="default rate")
ax2.set_ylabel("charge-off rate", color="darkred")
ax1.set_xlabel("US unemployment rate at issuance (%)")
ax1.set_title("Insight 1: Default rate vs unemployment at issuance")
plt.tight_layout()
plt.show()
q1

## Insight 2 - Grade x macro regime interaction

Does the relationship between loan grade and default strengthen in tighter rate environments? Split loans by `FEDFUNDS` regime and plot default rate by `sub_grade`.

In [ ]:
q2 = spark.sql(f"""
WITH t AS (
  SELECT
    sub_grade,
    CASE WHEN FEDFUNDS <= 0.5 THEN 'low rate (<=0.5%)'
         ELSE 'high rate (>0.5%)'
    END AS rate_regime,
    charged_off
  FROM {GOLD_COMBINED_TABLE}
  WHERE sub_grade IS NOT NULL AND FEDFUNDS IS NOT NULL
)
SELECT sub_grade, rate_regime, AVG(charged_off) AS default_rate, COUNT(*) AS n
FROM t
GROUP BY sub_grade, rate_regime
ORDER BY sub_grade, rate_regime
""").toPandas()

pivot2 = q2.pivot(index="sub_grade", columns="rate_regime", values="default_rate")
ax = pivot2.plot(kind="bar", figsize=(11, 5), color=["#4c72b0", "#c44e52"])
ax.set_ylabel("Charge-off rate")
ax.set_xlabel("Loan sub-grade")
ax.set_title("Insight 2: Default rate by sub-grade, split by Fed Funds regime")
plt.tight_layout()
plt.show()
pivot2

## Insight 3 - State-level unemployment differential and defaults

For borrowers in CA / TX / NY / FL / IL, does the gap between their state's unemployment rate and the national rate predict defaults? Compute `state_unrate - UNRATE` at issuance.

In [ ]:
q3 = spark.sql(f"""
SELECT
    addr_state,
    ROUND(state_unrate - UNRATE, 1) AS excess_unemp,
    COUNT(*)          AS n,
    AVG(charged_off)  AS default_rate
FROM {GOLD_COMBINED_TABLE}
WHERE addr_state IN ('CA','TX','NY','FL','IL')
  AND state_unrate IS NOT NULL AND UNRATE IS NOT NULL
GROUP BY addr_state, ROUND(state_unrate - UNRATE, 1)
ORDER BY addr_state, excess_unemp
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 5))
for st, g in q3.groupby("addr_state"):
    ax.plot(g["excess_unemp"], g["default_rate"], "o-", label=st, alpha=0.8)
ax.axvline(0, color="k", lw=0.5, ls="--")
ax.set_xlabel("State unemployment rate - National rate (percentage points)")
ax.set_ylabel("Charge-off rate")
ax.set_title("Insight 3: State unemployment differential vs default rate")
ax.legend()
plt.tight_layout()
plt.show()
q3.head(20)

## Step 5 - Retrain model with combined features

Same sklearn pipeline as notebook 04, now with macro features added. Compare Val AUC-ROC to the loan-only Phase 3 baseline (0.7136).

In [ ]:
combined = spark.table(GOLD_COMBINED_TABLE).withColumn("issue_year", F.year("issue_d"))
test = combined.filter(F.col("issue_year") == 2017)
trainval = combined.filter(F.col("issue_year").isin(2014, 2015, 2016))

from pyspark.sql.window import Window
w = Window.orderBy("issue_d")
trainval = trainval.withColumn("_rank", F.percent_rank().over(w))
train = trainval.filter(F.col("_rank") < 0.80).drop("_rank")
val   = trainval.filter(F.col("_rank") >= 0.80).drop("_rank")

def strat(df, n, seed=42):
    total = df.count()
    if total <= n:
        return df
    frac = n / total
    return df.stat.sampleBy("charged_off", fractions={0: frac, 1: frac}, seed=seed)

train = strat(train, 200_000)
val = strat(val, 50_000)

train_df = train.drop("issue_year").toPandas()
val_df   = val.drop("issue_year").toPandas()
test_df  = test.drop("issue_year").toPandas()
print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

In [ ]:
LABEL = "charged_off"
DROP_FOR_FEATURES = {LABEL, "issue_d", "earliest_cr_line", "year_month"}

numeric_cols = [c for c in train_df.columns
                if c not in DROP_FOR_FEATURES and pd.api.types.is_numeric_dtype(train_df[c])]
categorical_cols = [c for c in train_df.columns
                    if c not in DROP_FOR_FEATURES
                    and not pd.api.types.is_numeric_dtype(train_df[c])
                    and not pd.api.types.is_datetime64_any_dtype(train_df[c])]

macro_features = ["UNRATE", "FEDFUNDS", "CPIAUCSL", "CPI_YOY", "state_unrate", "real_int_rate", "rate_spread"]
print(f"Macro features in model: {[m for m in macro_features if m in numeric_cols]}")
print(f"Total numeric: {len(numeric_cols)}  |  categorical: {len(categorical_cols)}")

def Xy(df):
    return df[numeric_cols + categorical_cols], df[LABEL].astype(int)

X_train, y_train = Xy(train_df)
X_val,   y_val   = Xy(val_df)
X_test,  y_test  = Xy(test_df)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import time

preproc = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True))]), categorical_cols),
])
pipe = Pipeline([("preproc", preproc),
                 ("clf", LogisticRegression(max_iter=200, class_weight="balanced", n_jobs=-1))])

t0 = time.time(); pipe.fit(X_train, y_train); print(f"Fit time: {time.time() - t0:.1f}s")

def score(X, y, name):
    p = pipe.predict_proba(X)[:, 1]
    roc, pr = roc_auc_score(y, p), average_precision_score(y, p)
    print(f"{name:<8} AUC-ROC: {roc:.4f} | AUC-PR: {pr:.4f}")
    return roc, pr

train_roc, train_pr = score(X_train, y_train, "Train")
val_roc,   val_pr   = score(X_val,   y_val,   "Val")
test_roc,  test_pr  = score(X_test,  y_test,  "Test")

## Final comparison: loan-only vs loan + macro

In [ ]:
comparison = pd.DataFrame([
    {"Model": "Phase 2 LogisticRegression (sklearn, full data, loan features only)",
     "Val AUC-ROC": 0.7121, "Val AUC-PR": 0.3757},
    {"Model": "Phase 3 LogisticRegression (sklearn on Gold, loan features only)",
     "Val AUC-ROC": 0.7136, "Val AUC-PR": 0.3780},
    {"Model": "Phase 3 LogisticRegression (loan + FRED macro features)",
     "Val AUC-ROC": round(val_roc, 4), "Val AUC-PR": round(val_pr, 4)},
])
comparison